# BT-UNet Evaluation Pipeline
Loads saved models and evaluates them against train, validation, and test splits using the same pipeline as the training notebooks.

## Configuration
Edit `MODELS_TO_EVALUATE` and `MODEL_DIR` to match your saved model filenames and location.

In [1]:
# ============================================================
# CONFIGURATION — edit these to match your saved models
# ============================================================


MODELS_TO_EVALUATE = [
    "model_BT-UNet_5pct.keras",
    "model_BT-UNet_10pct.keras",
    "model_BT-UNet_20pct.keras",
    "model_BT-UNet_50pct.keras",
    "model_SimSiam-UNET_5.keras",
    "model_SimSiam-UNET_10.keras",
    "model_SimSiam-UNET_20.keras",
    "model_SimSiam-UNET_50.keras",
    "model_UNet_5pct.keras",
    "model_UNet_10pct.keras",
    "model_UNet_20pct.keras",
    "model_UNet_50pct.keras",
]

MODEL_DIR     = "saved_models"   # folder where .keras files live
THRESHOLD     = 0.5              # binarisation threshold
TRAIN_PATH = "datasets/KDSB/train/"
TEST_PATH  = "datasets/KDSB/test/"
IMG_HEIGHT    = 256
IMG_WIDTH     = 256
IMG_CHANNELS  = 3
val_split     = 0.3              # must match the split used during training

## Imports

In [8]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import re

from skimage.io import imread
from skimage.transform import resize
from skimage.morphology import label

import tensorflow as tf
from tensorflow.keras.metrics import Accuracy, Precision, Recall, MeanIoU, MeanAbsoluteError
from tensorflow.keras import backend as K
from tensorflow.keras.models import load_model
from hausdorff import hausdorff_distance

## Data Loading
Identical to the training notebooks.

In [3]:
SAVE_DIR = "saved_np"  # change if you want to save elsewhere

x_train_path = os.path.join(SAVE_DIR, "KDSB_X_train_256x256.npy")
y_train_path = os.path.join(SAVE_DIR, "KDSB_Y_train_256x256.npy")
x_test_path  = os.path.join(SAVE_DIR, "KDSB_X_test_256x256.npy")
y_test_path  = os.path.join(SAVE_DIR, "KDSB_Y_test_256x256.npy")

if all(os.path.exists(p) for p in [x_train_path, y_train_path, x_test_path, y_test_path]):
    print("Found cached arrays, loading...")
    X_train = np.load(x_train_path)
    Y_train = np.load(y_train_path)
    X_test  = np.load(x_test_path)
    Y_test  = np.load(y_test_path)
    print(f"Loaded — Train: {X_train.shape}, Test: {X_test.shape}")

else:
    print("No cached arrays found, loading and resizing from disk...")
    train_ids = next(os.walk(TRAIN_PATH + 'org/'))[2][:]
    test_ids  = next(os.walk(TEST_PATH  + 'org/'))[2][:]
    print(f"Train images: {len(train_ids)}, Test images: {len(test_ids)}")

    X_train = np.zeros((len(train_ids), IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS), dtype=np.float32)
    Y_train = np.zeros((len(train_ids), IMG_HEIGHT, IMG_WIDTH, 1),            dtype=np.float32)
    X_test  = np.zeros((len(test_ids),  IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS), dtype=np.float32)
    Y_test  = np.zeros((len(test_ids),  IMG_HEIGHT, IMG_WIDTH, 1),            dtype=np.float32)

    for n, id_ in tqdm(enumerate(train_ids), total=len(train_ids)):
        img = imread(TRAIN_PATH + 'org/' + id_)[:, :, :IMG_CHANNELS]
        img = resize(img, (IMG_HEIGHT, IMG_WIDTH), mode='constant', preserve_range=True)
        X_train[n] = img
        mask = imread(TRAIN_PATH + 'gt/' + id_[:-4] + '_GT.png')
        mask = np.expand_dims(mask, axis=-1)
        mask = resize(mask, (IMG_HEIGHT, IMG_WIDTH, 1), mode='constant', preserve_range=True)
        Y_train[n][mask[:, :, 0] / 255 > 0.] = 1.

    for n, id_ in tqdm(enumerate(test_ids), total=len(test_ids)):
        img = imread(TEST_PATH + 'org/' + id_)[:, :, :IMG_CHANNELS]
        img = resize(img, (IMG_HEIGHT, IMG_WIDTH), mode='constant', preserve_range=True)
        X_test[n] = img
        mask = imread(TEST_PATH + 'gt/' + id_[:-4] + '_GT.png')
        mask = np.expand_dims(mask, axis=-1)
        mask = resize(mask, (IMG_HEIGHT, IMG_WIDTH, 1), mode='constant', preserve_range=True)
        Y_test[n][mask[:, :, 0] / 255 > 0.] = 1.

    np.save(x_train_path, X_train)
    np.save(y_train_path, Y_train)
    np.save(x_test_path,  X_test)
    np.save(y_test_path,  Y_test)
    print(f"Saved — Train: {X_train.shape}, Test: {X_test.shape}")

Found cached arrays, loading...
Loaded — Train: (2594, 256, 256, 3), Test: (1000, 256, 256, 3)


In [6]:
# Recreate train/val split — must match training notebooks
X_train_20    = X_train[:int(X_train.shape[0] * 0.5)]
Y_train_20    = Y_train[:int(Y_train.shape[0] * 0.5)]

split_idx     = int(X_train_20.shape[0] * (1 - val_split))
X_train_split = X_train_20[:split_idx]
Y_train_split = Y_train_20[:split_idx]
X_val         = X_train_20[split_idx:]
Y_val         = Y_train_20[split_idx:]

print(f"X_train_split: {X_train_split.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")

X_train_split: (907, 256, 256, 3), X_val: (390, 256, 256, 3), X_test: (1000, 256, 256, 3)


## Metric Helpers
Identical to the training notebooks.

In [9]:
def iou_metric(y_true_in, y_pred_in, print_table=False):
    labels = label(y_true_in > 0.5)
    y_pred = label(y_pred_in > 0.5)
    true_objects = len(np.unique(labels))
    pred_objects = len(np.unique(y_pred))
    intersection = np.histogram2d(labels.flatten(), y_pred.flatten(),
                                  bins=(true_objects, pred_objects))[0]
    area_true = np.expand_dims(np.histogram(labels, bins=true_objects)[0], -1)
    area_pred = np.expand_dims(np.histogram(y_pred, bins=pred_objects)[0],  0)
    union = area_true + area_pred - intersection
    intersection = intersection[1:, 1:]
    union = union[1:, 1:]
    union[union == 0] = 1e-9
    iou = intersection / union

    def precision_at(threshold, iou):
        matches = iou > threshold
        tp = np.sum(np.sum(matches, axis=1) == 1)
        fp = np.sum(np.sum(matches, axis=0) == 0)
        fn = np.sum(np.sum(matches, axis=1) == 0)
        return tp, fp, fn

    prec = []
    for t in np.arange(0.5, 1.0, 0.05):
        tp, fp, fn = precision_at(t, iou)
        p = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
        prec.append(p)
    return np.mean(prec)


def iou_metric_batch(y_true_in, y_pred_in):
    batch_size = y_true_in.shape[0]
    value = 0.
    for batch in range(batch_size):
        value += iou_metric(y_true_in[batch], y_pred_in[batch])
    return value / batch_size


def dice_coeff(y_true, y_pred):
    smooth = 1.
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)


def haud_dist(y_true, y_pred):
    return hausdorff_distance(np.squeeze(y_true), np.squeeze(y_pred))


def haud_dist_batch(y_true, y_pred):
    if len(y_true.shape) == 2:
        return haud_dist(y_true, y_pred)
    hd = 0.
    for batch in range(y_true.shape[0]):
        hd += haud_dist(y_true[batch], y_pred[batch])
    return hd / y_true.shape[0]


def evalResult(gt, pred, num_class=2):
    """Identical computation to training notebooks; also returns a metrics dict."""
    gt   = np.squeeze(gt)
    pred = np.squeeze(pred)

    acc = Accuracy()
    acc.update_state(gt, pred)
    r_acc = acc.result().numpy()

    pr = Precision()
    pr.update_state(gt, pred)
    r_pr = pr.result().numpy()

    rc = Recall()
    rc.update_state(gt, pred)
    r_rc = rc.result().numpy()

    mi = MeanIoU(num_class)
    mi.update_state(gt, pred)
    r_mi = mi.result().numpy()

    dc = 0.
    for img in range(gt.shape[0]):
        dc += dice_coeff(gt[img], pred[img]).numpy()
    dc /= gt.shape[0]

    hd   = haud_dist_batch(gt, pred)
    miou = iou_metric_batch(gt, pred)

    mae = MeanAbsoluteError()
    r_mae = mae(gt, pred).numpy()

    print(f"  Accuracy={r_acc:.4f}  Precision={r_pr:.4f}  Recall={r_rc:.4f}  "
          f"MeanIoU={r_mi:.4f}  Dice={dc:.4f}  HD={hd:.4f}  MyIoU={miou:.4f}  MAE={r_mae:.4f}")

    return {
        "Accuracy":  r_acc,
        "Precision": r_pr,
        "Recall":    r_rc,
        "MeanIoU":   r_mi,
        "Dice":      dc,
        "HD":        hd,
        "MyIoU":     miou,
        "MAE":       r_mae,
    }
    
def get_label_fraction(model_name):
    """Extract label fraction from model filename. Returns a float between 0 and 1."""
    match = re.search(r'_(\d+)(?:pct)?\.keras', model_name)
    if not match:
        raise ValueError(f"Could not parse label fraction from: {model_name}")
    return int(match.group(1)) / 100.0

## Evaluation Loop

In [10]:
results = []

for model_name in MODELS_TO_EVALUATE:

    model_path = os.path.join(MODEL_DIR, model_name)

    if not os.path.exists(model_path):
        print(f"\n[SKIP] Model not found: {model_path}")
        continue

    # ---- Derive the correct split for this model ----
    label_fraction = get_label_fraction(model_name)
    n_labeled      = int(X_train.shape[0] * label_fraction)
    X_subset       = X_train[:n_labeled]
    Y_subset       = Y_train[:n_labeled]

    split_idx     = int(n_labeled * (1 - val_split))
    X_train_split = X_subset[:split_idx]
    Y_train_split = Y_subset[:split_idx]
    X_val         = X_subset[split_idx:]
    Y_val         = Y_subset[split_idx:]

    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name}  |  label fraction: {label_fraction:.0%}")
    print(f"  train={X_train_split.shape[0]}  val={X_val.shape[0]}  test={X_test.shape[0]}")
    print(f"{'='*60}")

    model = load_model(model_path, compile=False)

    # ---- Predictions ----
    preds_train = model.predict(X_train_split, verbose=1)
    preds_val   = model.predict(X_val,         verbose=1)
    preds_test  = model.predict(X_test,        verbose=1)

    # ---- Threshold ----
    preds_train_t = (preds_train > THRESHOLD).astype(np.float32)
    preds_val_t   = (preds_val   > THRESHOLD).astype(np.float32)
    preds_test_t  = (preds_test  > THRESHOLD).astype(np.float32)

    # ---- Evaluate ----
    print("\nTRAIN")
    train_m = evalResult(Y_train_split.astype(np.float32), preds_train_t)

    print("\nVALIDATION")
    val_m = evalResult(Y_val.astype(np.float32), preds_val_t)

    print("\nTEST")
    test_m = evalResult(Y_test.astype(np.float32), preds_test_t)

    # ---- Collect ----
    row = {"Model": model_name, "Label Fraction": f"{label_fraction:.0%}"}
    for split_name, metrics in [("Train", train_m), ("Val", val_m), ("Test", test_m)]:
        for k, v in metrics.items():
            row[f"{split_name} {k}"] = v
    results.append(row)

print("\nDone.")


Evaluating: model_BT-UNet_5pct.keras  |  label fraction: 5%
  train=90  val=39  test=1000
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 603ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 12s 369ms/step

TRAIN
  Accuracy=0.8156  Precision=0.8286  Recall=0.4946  MeanIoU=0.6159  Dice=0.6236  HD=7.2282  MyIoU=0.0352  MAE=0.1844

VALIDATION
  Accuracy=0.8746  Precision=0.9121  Recall=0.5782  MeanIoU=0.6999  Dice=0.6948  HD=6.3248  MyIoU=0.0270  MAE=0.1254

TEST
  Accuracy=0.7926  Precision=0.7195  Recall=0.4898  MeanIoU=0.5844  Dice=0.5433  HD=8.2390  MyIoU=0.0220  MAE=0.2074

Evaluating: model_BT-UNet_10pct.keras  |  label fraction: 10%
  train=181  val=78  test=1000
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 445ms/step
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 264ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 12s 371ms/step

TRAIN
  Accuracy=0.9606  Precision=0.9427  Recall=0.9291  MeanIoU=0.9121  Dice=0.9286  HD=4.7818  MyIoU=0.7402  MAE=0.0394

VALIDATION
  Accuracy=0.9440  Precision=0.9666  Recall=0.8979  MeanIoU=0.89

In [23]:
print(preds_train_t.shape)
print(Y_train_split.shape)

(907, 128)
(907, 256, 256, 1)


## Results Table

In [11]:
results_df = pd.DataFrame(results)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

print("\n================ FINAL RESULTS ================\n")
display(results_df)

results_df.to_csv("evaluation_results.csv", index=False)
print("\nSaved to evaluation_results.csv")


================ FINAL RESULTS ================



,Model,Label Fraction,Train Accuracy,Train Precision,Train Recall,Train MeanIoU,Train Dice,Train HD,Train MyIoU,Train MAE,Val Accuracy,Val Precision,Val Recall,Val MeanIoU,Val Dice,Val HD,Val MyIoU,Val MAE,Test Accuracy,Test Precision,Test Recall,Test MeanIoU,Test Dice,Test HD,Test MyIoU,Test MAE
0,model_BT-UNet_5pct.keras,5%,0.8156,0.8286,0.4946,0.6159,0.6236,7.2282,0.0352,0.1844,0.8746,0.9121,0.5782,0.6999,0.6948,6.3248,0.0270,0.1254,0.7926,0.7195,0.4898,0.5844,0.5433,8.2390,0.0220,0.2074
1,model_BT-UNet_10pct.keras,10%,0.9606,0.9427,0.9291,0.9121,0.9286,4.7818,0.7402,0.0394,0.9440,0.9666,0.8979,0.8904,0.9266,5.2077,0.7350,0.0560,0.8093,0.6322,0.8502,0.6570,0.7327,7.8514,0.3071,0.1907
2,model_BT-UNet_20pct.keras,20%,0.9662,0.9758,0.9223,0.9263,0.9452,4.5531,0.8226,0.0338,0.9086,0.7722,0.8916,0.7944,0.7843,6.4875,0.5261,0.0914,0.8953,0.8171,0.8322,0.7813,0.8309,6.1099,0.5272,0.1047
3,model_BT-UNet_50pct.keras,50%,0.9724,0.9604,0.9407,0.9341,0.9440,4.3677,0.8049,0.0276,0.9233,0.8430,0.8468,0.8173,0.8312,5.5984,0.5274,0.0767,0.9082,0.8576,0.8271,0.8028,0.8431,5.7391,0.4922,0.0918
4,model_SimSiam-UNET_5.keras,5%,0.8700,0.7648,0.8253,0.7424,0.7876,6.9447,0.2196,0.1300,0.8305,0.6188,0.9228,0.6824,0.7287,8.1240,0.1045,0.1695,0.6294,0.4360,0.8603,0.4550,0.5738,10.8779,0.0674,0.3706
5,model_SimSiam-UNET_10.keras,10%,0.9626,0.9506,0.9274,0.9162,0.9308,4.5960,0.7557,0.0374,0.9263,0.9633,0.8576,0.8576,0.9035,5.6282,0.6420,0.0737,0.8469,0.7381,0.7476,0.6971,0.7396,6.9883,0.3397,0.1531
6,model_SimSiam-UNET_20.keras,20%,0.9777,0.9802,0.9529,0.9510,0.9617,4.1935,0.9004,0.0223,0.9076,0.7745,0.8813,0.7917,0.7822,6.4374,0.5162,0.0924,0.8860,0.8083,0.8059,0.7635,0.8107,6.2958,0.4997,0.1140
7,model_SimSiam-UNET_50.keras,50%,0.9751,0.9713,0.9393,0.9401,0.9486,4.2241,0.8479,0.0249,0.9256,0.8659,0.8261,0.8195,0.8418,5.4202,0.5671,0.0744,0.9104,0.8718,0.8175,0.8058,0.8444,5.6757,0.5347,0.0896
8,model_UNet_5pct.keras,5%,0.7023,0.5520,0.0996,0.3926,0.1820,10.2545,0.0050,0.2977,0.7577,0.7155,0.1280,0.4356,0.2354,9.2614,0.0021,0.2423,0.7367,0.8431,0.1352,0.4288,0.2013,9.8089,0.0106,0.2633
9,model_UNet_10pct.keras,10%,0.9802,0.9737,0.9620,0.9547,0.9639,3.9172,0.9006,0.0198,0.9350,0.9722,0.8705,0.8733,0.9186,5.3689,0.7015,0.0650,0.8599,0.7530,0.7834,0.7204,0.7720,6.7466,0.3927,0.1401



Saved to evaluation_results.csv


## Per-metric Summary (Test set)

In [ ]:
if not results_df.empty:
    test_cols = [c for c in results_df.columns if c.startswith("Test")]
    display(results_df[["Model"] + test_cols])